In [8]:
# from database import get_connection, release_connection, create_tables
import psycopg2
import psycopg2.extras
import pandas as pd

In [80]:
connection = psycopg2.connect(database="mydb", user="myuser", password="mypassword", host="localhost", port=5433)

cursor = connection.cursor()

### Load in dataframes

In [60]:
# grammar_df = pd.read_csv('../../grammar_points.csv', encoding='utf-16', index_col=0, names=["level", "grammar_point_jp", "meaning_en", "description", "url"])
# print(grammar_df)

# example_df = pd.read_csv('../../examples.csv', encoding='utf-16', index_col=0, names=["grammar_id", "text_jp", "text_en", "grammar_char_locations_jp"])
# example_df

### Insert batch data

In [62]:
cursor.execute("""
    INSERT INTO level (id, name) VALUES 
               (1, 'n1'), 
               (2, 'n2'),
               (3, 'n3'),
               (4, 'n4'),
               (5, 'n5'),
               (6, 'NA');
               """)

connection.commit()


In [81]:
f = open(r'C:\\Users\\mcurt\\JapaneseTranslationActivity\\grammar_points.csv', 'r', encoding='utf-8')

cursor.copy_expert(
    "COPY grammar FROM STDIN DELIMITER ',' CSV ENCODING 'UTF8'",
    f
    )

f = open(r'C:\\Users\\mcurt\\JapaneseTranslationActivity\\examples.csv', 'r', encoding='utf-8')

cursor.copy_expert(
    "COPY example FROM STDIN DELIMITER ',' CSV ENCODING 'UTF8'",
    f
    )

connection.commit()

### Turn the database string representation of text into a list of character metadata

In [ ]:
import numpy as np
import json

def stringToList(string: str):
    # Returns text metadata retrieved from database as a list
    # Format of text metadata is [[kanji: str, furigana: str, hiragana : str, grammar : Bool]]
    list = []
    i = 1
    start_of_quote = False
    word = ''
    while i < len(string)-1:

        char = string[i] 
        
        if char == '[':
            sub_list = []
        elif char == ']':
            list.append(sub_list)
        elif char == ',' and not start_of_quote:
            pass
        elif char == " ":
            pass
        else:
            if char == "N": # Always equals None
                sub_list.append(None)
                i += 3
            elif char == "T":
                sub_list.append(True)
                i += 3
            elif char == "F":
                sub_list.append(False)
                i += 4
            elif char == "'":
                if not start_of_quote: # If char is the opening quotation mark
                    word = '' 
                else:
                    sub_list.append(word)
                start_of_quote = not start_of_quote
            else: # Any letters in a quote
                word += char
                
        i += 1

    return list
        

In [140]:

cursor.execute("SELECT id, grammar, example_jp, example_en, highlight_indices FROM example where id = 0;")

result = cursor.fetchall()

print(result[0][2])

list = stringToList(result[0][2])
print(list)
print(len(list))

[['私', 'わたし', None, None], [None, None, 'だ', True], [None, None, '。', None]]
[['私', 'わたし', None, None], [None, None, 'だ', True], [None, None, '。', None]]
3
